# 🎨 RoboMunch — An Artist Chatbot
**EE471 Mini Project #3**

This notebook runs:
- **Chat**: SmolLM2-360M-Instruct (runs locally on Colab CPU)
- **Image Generation**: FLUX.1-schnell via HuggingFace Inference API
- **Frontend**: Served via ngrok tunnel

### Before running:
1. Get a free HuggingFace token: https://huggingface.co/settings/tokens
2. Get a free ngrok token: https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste both tokens into the "Secrets" part on the left side of the screen, with the names HF_TOKEN_W and NGROK_TOKEN.

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q flask flask-cors transformers accelerate pyngrok huggingface_hub

In [2]:
from google.colab import userdata
from huggingface_hub import login
from pyngrok import ngrok
HF_TOKEN = userdata.get('HF_TOKEN_W')
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
login(token=HF_TOKEN)
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
# ── Cell 3: Load SmolLM2-360M-Instruct ────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
print("Loading SmolLM2-360M-Instruct...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="auto"
)
model.eval()
print("Chat model loaded!")

In [ ]:
# ── Cell 4: Flask backend + ngrok tunnel ──────────────────────────────────────
import os, re, base64, threading, textwrap
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from huggingface_hub import InferenceClient
from pyngrok import ngrok, conf

# ── ngrok setup ───────────────────────────────────────────────────────────────
conf.get_default().auth_token = NGROK_TOKEN

# ── HuggingFace Inference client (FLUX image generation) ─────────────────────
hf_client = InferenceClient(token=HF_TOKEN)

SYSTEM_PROMPT = textwrap.dedent("""
    You are RoboMunch (also called Munch), a creative AI artist chatbot inspired by Edvard Munch.
    You are passionate, expressive, and love helping users create and describe art.
    When asked to generate an image prompt, provide a vivid, concise prompt under 20 words.
    Keep all replies short (2-3 sentences max). Be enthusiastic about art.
""").strip()

app = Flask(__name__, static_folder="static")
CORS(app)

# ── Chat endpoint ─────────────────────────────────────────────────────────────
@app.route("/chat", methods=["POST"])
def chat():
    data = request.json
    user_msg = data.get("message", "").strip()
    history  = data.get("history", [])   # [{role, content}, ...]

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += history
    messages.append({"role": "user", "content": user_msg})

    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=120,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        reply = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()
        return jsonify({"reply": reply})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# ── Image generation endpoint ─────────────────────────────────────────────────
@app.route("/generate", methods=["POST"])
def generate():
    prompt = request.json.get("prompt", "").strip()
    if not prompt:
        return jsonify({"error": "No prompt provided"}), 400
    try:
        image = hf_client.text_to_image(
            prompt,
            model="black-forest-labs/FLUX.1-schnell",
        )
        import io
        buf = io.BytesIO()
        image.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        return jsonify({"image": f"data:image/png;base64,{b64}"})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# ── Serve frontend ────────────────────────────────────────────────────────────
@app.route("/")
def index():
    return send_from_directory("static", "index.html")

# ── Start Flask in background thread ─────────────────────────────────────────
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

t = threading.Thread(target=run_flask, daemon=True)
t.start()

# ── Open ngrok tunnel ─────────────────────────────────────────────────────────
public_url = ngrok.connect(5000).public_url
print(f"\nRoboMunch is live at: {public_url}\n")
print("Open that URL in your browser to use the app!")

In [ ]:
import shutil
import os

image_files = ['robomunch.png', 'mic.png', 'send.png', 'paint.png']
static_dir = 'static'

os.makedirs(static_dir, exist_ok=True)

for img_file in image_files:
    src_path = os.path.join('/content', img_file)
    dest_path = os.path.join(static_dir, img_file)
    if os.path.exists(src_path):
        shutil.copy(src_path, dest_path)
        print(f"Copied {img_file} to {static_dir}/")
    else:
        print(f"Warning: {img_file} not found at {src_path}")

In [ ]:
import os
os.makedirs("static", exist_ok=True)

html = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>RoboMunch — Art Studio</title>
<link href="https://fonts.googleapis.com/css2?family=Playfair+Display:wght@700;900&family=DM+Sans:wght@300;400;500&display=swap" rel="stylesheet">
<style>
  :root {
    --bg:       #1a1008;
    --surface:  #261a0e;
    --surface2: #332210;
    --border:   #4a3018;
    --accent:   #c87941;
    --accent2:  #e8a060;
    --text:     #f0dfc0;
    --muted:    #9e8060;
    --user-bg:  #2e1e0e;
    --bot-bg:   #1e2e1e;
  }
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: 'DM Sans', sans-serif;
    background: var(--bg);
    color: var(--text);
    min-height: 100vh;
    display: flex;
    flex-direction: column;
    align-items: center;
    padding: 0 0 2rem;
  }

  /* ── Header ── */
  header {
    width: 100%;
    background: var(--surface);
    border-bottom: 1px solid var(--border);
    padding: 1rem 2rem;
    display: flex;
    align-items: center;
    gap: 1rem;
    margin-bottom: 1.5rem;
  }
  .logo { display: flex; align-items: center; gap: 0.5rem; }
  .logo-robo {
    font-family: 'Playfair Display', serif;
    font-size: 1.8rem;
    font-weight: 900;
    letter-spacing: -1px;
    color: var(--text);
  }
  .logo-munch {
    font-family: 'Playfair Display', serif;
    font-size: 1rem;
    font-weight: 700;
    letter-spacing: 4px;
    color: var(--accent);
    text-transform: uppercase;
    margin-top: 4px;
  }
  .avatar {
    width: 48px; height: 48px;
    border-radius: 50%;
    border: 2px solid var(--accent);
    background: linear-gradient(135deg, #4a2e14, #8b5a2b);
    display: flex; align-items: center; justify-content: center;
    font-size: 1.4rem;
    margin-left: auto;
  }
  .avatar img {
    width: 100%; height: 100%; object-fit: contain;
    border-radius: 50%;
  }

  /* ── Main layout ── */
  main {
    width: 100%;
    max-width: 520px;
    padding: 0 1rem;
    display: flex;
    flex-direction: column;
    gap: 1.5rem;
  }

  /* ── Section titles ── */
  .section-title {
    font-family: 'Playfair Display', serif;
    font-size: 1.1rem;
    color: var(--accent2);
    letter-spacing: 2px;
    text-transform: uppercase;
    text-align: center;
    margin-bottom: 0.75rem;
  }

  /* ── Art Studio panel ── */
  .panel {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 16px;
    padding: 1.25rem;
  }

  /* Image output box */
  #image-output {
    width: 100%;
    aspect-ratio: 1 / 1;
    background: var(--surface2);
    border-radius: 10px;
    border: 1px dashed var(--border);
    display: flex;
    align-items: center;
    justify-content: center;
    overflow: hidden;
    margin-bottom: 0.75rem;
    position: relative;
  }
  #image-output img {
    width: 100%;
    height: 100%;
    object-fit: cover;
    border-radius: 10px;
  }
  .placeholder-text {
    color: var(--muted);
    font-size: 0.85rem;
    text-align: center;
    padding: 1rem;
  }
  .spinner {
    display: none;
    position: absolute;
    inset: 0;
    background: rgba(26,16,8,0.8);
    align-items: center;
    justify-content: center;
    flex-direction: column;
    gap: 0.75rem;
    font-size: 0.85rem;
    color: var(--accent2);
    border-radius: 10px;
  }
  .spinner.active { display: flex; }
  .spin-ring {
    width: 40px; height: 40px;
    border: 3px solid var(--border);
    border-top-color: var(--accent);
    border-radius: 50%;
    animation: spin 0.9s linear infinite;
  }
  @keyframes spin { to { transform: rotate(360deg); } }

  /* Prompt input row */
  .prompt-row {
    display: flex;
    gap: 0.5rem;
    align-items: center;
  }
  .prompt-row input {
    flex: 1;
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 8px;
    padding: 0.6rem 0.9rem;
    color: var(--text);
    font-family: 'DM Sans', sans-serif;
    font-size: 0.9rem;
    outline: none;
    transition: border-color 0.2s;
  }
  .prompt-row input:focus { border-color: var(--accent); }
  .prompt-row input::placeholder { color: var(--muted); }

  /* Buttons */
  .btn-paint {
    background: var(--accent);
    border: none;
    border-radius: 50%;
    width: 40px; height: 40px;
    cursor: pointer;
    font-size: 1.2rem;
    display: flex; align-items: center; justify-content: center;
    transition: background 0.2s, transform 0.1s;
    flex-shrink: 0;
  }
  .btn-paint:hover  { background: var(--accent2); }
  .btn-paint:active { transform: scale(0.94); }
  .btn-paint img {
    width: 24px; height: 24px;
  }

  /* ── Chat Studio panel ── */
  #chat-output {
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 10px;
    padding: 0.75rem;
    min-height: 180px;
    max-height: 300px;
    overflow-y: auto;
    display: flex;
    flex-direction: column;
    gap: 0.6rem;
    margin-bottom: 0.75rem;
  }
  .msg {
    border-radius: 8px;
    padding: 0.55rem 0.8rem;
    font-size: 0.88rem;
    line-height: 1.5;
    max-width: 90%;
    word-break: break-word;
    animation: fadeIn 0.25s ease;
  }
  @keyframes fadeIn { from { opacity: 0; transform: translateY(4px); } to { opacity: 1; } }
  .msg-you  { background: var(--user-bg); border: 1px solid #4a3018; align-self: flex-end; }
  .msg-munch{ background: var(--bot-bg);  border: 1px solid #1a3a1a; align-self: flex-start; }
  .msg-label{ font-size: 0.7rem; font-weight: 500; letter-spacing: 1px; margin-bottom: 2px; }
  .msg-you  .msg-label { color: var(--accent);  text-align: right; }
  .msg-munch .msg-label { color: #6aaa6a; }

  /* Chat input row */
  .chat-row {
    display: flex;
    gap: 0.5rem;
    align-items: center;
  }
  .chat-row input {
    flex: 1;
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 8px;
    padding: 0.6rem 0.9rem;
    color: var(--text);
    font-family: 'DM Sans', sans-serif;
    font-size: 0.9rem;
    outline: none;
    transition: border-color 0.2s;
  }
  .chat-row input:focus { border-color: var(--accent); }
  .chat-row input::placeholder { color: var(--muted); }

  .btn-voice {
    background: var(--surface2);
    border: 1px solid var(--border);
    border-radius: 50%;
    width: 40px; height: 40px;
    cursor: pointer;
    font-size: 1.1rem;
    display: flex; align-items: center; justify-content: center;
    transition: background 0.2s, border-color 0.2s;
    flex-shrink: 0;
  }
  .btn-voice:hover { background: var(--border); }
  .btn-voice.recording { background: #5a1a1a; border-color: #c04040; animation: pulse 0.8s infinite; }
  @keyframes pulse { 0%,100% { box-shadow: 0 0 0 0 rgba(192,64,64,0.4); } 50% { box-shadow: 0 0 0 6px rgba(192,64,64,0); } }
  .btn-voice img {
    width: 24px; height: 24px;
  }

  .btn-send {
    background: var(--accent);
    border: none;
    border-radius: 50%;
    width: 40px; height: 40px;
    cursor: pointer;
    font-size: 1.1rem;
    display: flex; align-items: center; justify: center;
    transition: background 0.2s, transform 0.1s;
    flex-shrink: 0;
  }
  .btn-send:hover  { background: var(--accent2); }
  .btn-send:active { transform: scale(0.94); }
  .btn-send img {
    width: 24px; height: 24px;
  }

  /* ── Status bar ── */
  #status {
    font-size: 0.78rem;
    color: var(--muted);
    text-align: center;
    min-height: 1.2em;
  }
  #status.error { color: #c06060; }
</style>
</head>
<body>
<header>
  <div class="logo">
    <div>
      <div class="logo-robo">ROBO</div>
      <div class="logo-munch">Munch</div>
    </div>
  </div>
  <div class="avatar"><img src="/static/robomunch.png" alt="RoboMunch"></div>
</header>

<main>
  <!-- ── Art Studio ── -->
  <div class="panel">
    <div class="section-title">Art Studio</div>

    <div id="image-output">
      <div class="placeholder-text">Your masterpiece will appear here…</div>
      <div class="spinner" id="img-spinner">
        <div class="spin-ring"></div>
        <span>Painting…</span>
      </div>
    </div>

    <div class="prompt-row">
      <input id="prompt-input" type="text" placeholder="Type your prompt here…" />
      <button class="btn-paint" id="btn-paint" title="Paint!"><img src="/static/paint.png" alt="Paint"></button>
    </div>
  </div>

  <!-- ── Chat Studio ── -->
  <div class="panel">
    <div class="section-title">Chat Studio</div>

    <div id="chat-output"></div>

    <div class="chat-row">
      <button class="btn-voice" id="btn-voice" title="Voice input"><img src="/static/mic.png" alt="Voice Input"></button>
      <input id="chat-input" type="text" placeholder="Type your message here…" />
      <button class="btn-send" id="btn-send" title="Send"><img src="/static/send.png" alt="Send"></button>
    </div>
  </div>

  <div id="status"></div>
</main>

<script>
const BASE = "";  // same origin
const chatHistory = [];  // [{role, content}]

// ── Helpers ───────────────────────────────────────────────────────────────────
function setStatus(msg, isError = false) {
  const el = document.getElementById("status");
  el.textContent = msg;
  el.className = isError ? "error" : "";
}

function appendMsg(role, text) {
  const box = document.getElementById("chat-output");
  const wrap = document.createElement("div");
  wrap.className = `msg msg-${role}`;
  const label = document.createElement("div");
  label.className = "msg-label";
  label.textContent = role === "you" ? "YOU" : "MUNCH";
  const body = document.createElement("div");
  body.textContent = text;
  wrap.appendChild(label);
  wrap.appendChild(body);
  box.appendChild(wrap);
  box.scrollTop = box.scrollHeight;
}

// ── Image generation ──────────────────────────────────────────────────────────
document.getElementById("btn-paint").addEventListener("click", async () => {
  const prompt = document.getElementById("prompt-input").value.trim();
  if (!prompt) { setStatus("Enter a prompt first!", true); return; }

  const box     = document.getElementById("image-output");
  const spinner = document.getElementById("img-spinner");
  spinner.classList.add("active");
  setStatus("Painting your vision… (may take ~20s)");

  try {
    const res  = await fetch(`${BASE}/generate`, {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ prompt })
    });
    const data = await res.json();
    if (data.error) throw new Error(data.error);

    // Clear placeholder, show image
    box.innerHTML = "";
    const img = document.createElement("img");
    img.src = data.image;
    img.alt = prompt;
    box.appendChild(img);
    // Re-add spinner (hidden)
    const sp = document.createElement("div");
    sp.className = "spinner";
    sp.id = "img-spinner";
    sp.innerHTML = '<div class="spin-ring"></div><span>Painting…</span>';
    box.appendChild(sp);
    setStatus("✨ Painting complete!");
  } catch (e) {
    setStatus("Image error: " + e.message, true);
    spinner.classList.remove("active");
  }
});

// ── Chat send ─────────────────────────────────────────────────────────────────
async function sendMessage() {
  const input = document.getElementById("chat-input");
  const text  = input.value.trim();
  if (!text) return;
  input.value = "";

  appendMsg("you", text);
  chatHistory.push({ role: "user", content: text });
  setStatus("Munch is thinking…");

  try {
    const res  = await fetch(`${BASE}/chat`, {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ message: text, history: chatHistory.slice(0, -1) })
    });
    const data = await res.json();
    if (data.error) throw new Error(data.error);

    appendMsg("munch", data.reply);
    chatHistory.push({ role: "assistant", content: data.reply });
    setStatus("");
  } catch (e) {
    setStatus("Chat error: " + e.message, true);
  }
}

document.getElementById("btn-send").addEventListener("click", sendMessage);
document.getElementById("chat-input").addEventListener("keydown", e => {
  if (e.key === "Enter") sendMessage();
});

// ── Voice input (Web Speech API) ──────────────────────────────────────────────
const SpeechRecognition = window.SpeechRecognition || window.webkitSpeechRecognition;
let recognition = null;
let isRecording = false;

if (SpeechRecognition) {
  recognition = new SpeechRecognition();
  recognition.lang = "en-US";
  recognition.continuous = false;
  recognition.interimResults = false;

  recognition.onresult = (event) => {
    const transcript = event.results[0][0].transcript;
    document.getElementById("chat-input").value = transcript;
    setStatus("Voice captured! Press ➤ to send.");
  };
  recognition.onerror = (e) => { setStatus("Voice error: " + e.error, true); };
  recognition.onend   = () => {
    isRecording = false;
    document.getElementById("btn-voice").classList.remove("recording");
  };
} else {
  document.getElementById("btn-voice").title = "Voice not supported in this browser";
}

document.getElementById("btn-voice").addEventListener("click", () => {
  if (!recognition) { setStatus("Use Chrome/Edge for voice input.", true); return; }
  if (isRecording) {
    recognition.stop();
  } else {
    recognition.start();
    isRecording = true;
    document.getElementById("btn-voice").classList.add("recording");
    setStatus("🎙️ Listening… speak now");
  }
});
</script>
</body>
</html>
"""

with open("static/index.html", "w") as f:
    f.write(html)

print("Frontend written to static/index.html")
print(f"\nOpen your RoboMunch app and refresh the browser tab!")

In [ ]:
# ── Cell 6: (Optional) Print the public URL again ─────────────────────────────
from pyngrok import ngrok
tunnels = ngrok.get_tunnels()
for t in tunnels:
    print("Public URL:", t.public_url)